In [9]:
!pip install yfinance pandas numpy requests --quiet

In [17]:
import requests
import pandas as pd
import numpy as np
import time
import json
import yfinance as yf
from datetime import datetime, timedelta

SEC_HEADERS = {
    "User-Agent": "GeneralMills ML Project srisanthoshnagan@webster.edu.edu",
    "Accept-Encoding": "gzip, deflate",
    "Host": "data.sec.gov"
}

GIS_CIK = "CIK0000040704"
GIS_TICKER = "GIS"

print("Configuration loaded")
print(f"CIK: {GIS_CIK}")
print(f"User-Agent: {SEC_HEADERS['User-Agent']}")

Configuration loaded
CIK: CIK0000040704
User-Agent: GeneralMills ML Project srisanthoshnagan@webster.edu.edu


In [18]:
url = f"https://data.sec.gov/api/xbrl/companyfacts/{GIS_CIK}.json"
print(f"Fetching from SEC: {url}")

response = requests.get(url, headers=SEC_HEADERS)
time.sleep(0.5)

if response.status_code == 200:
    company_facts = response.json()
    print("✅ SEC data retrieved successfully!")
    print(f"Company name: {company_facts.get('entityName', 'N/A')}")
    print(f"CIK: {company_facts.get('cik', 'N/A')}")
else:
    print(f"Error: {response.status_code}")

Fetching from SEC: https://data.sec.gov/api/xbrl/companyfacts/CIK0000040704.json
✅ SEC data retrieved successfully!
Company name: General Mills, Inc.
CIK: 40704


In [19]:
def extract_concept(facts_json, concept):
    try:
        units = facts_json['facts']['us-gaap'][concept]['units']
        unit_key = list(units.keys())[0]
        records = units[unit_key]
        df = pd.DataFrame(records)
        df['end'] = pd.to_datetime(df['end'])
        df['filed'] = pd.to_datetime(df['filed'])
        df = df[df['form'].isin(['10-K', '10-Q'])]
        df = df.sort_values('filed').drop_duplicates(subset=['end'], keep='last')
        df = df[['end', 'val', 'form', 'filed']].rename(
            columns={'val': concept, 'end': 'period_end'}
        )
        return df.reset_index(drop=True)
    except KeyError:
        print(f"Could not find: {concept}")
        return pd.DataFrame()

revenue_df   = extract_concept(company_facts, 'Revenues')
netincome_df = extract_concept(company_facts, 'NetIncomeLoss')
assets_df    = extract_concept(company_facts, 'Assets')
debt_df      = extract_concept(company_facts, 'LongTermDebt')
opincome_df  = extract_concept(company_facts, 'OperatingIncomeLoss')

print("✅ Extracted all financial concepts!")
print(f"Revenue rows: {len(revenue_df)}")
print(f"Net Income rows: {len(netincome_df)}")
print(f"Assets rows: {len(assets_df)}")
print(f"Long Term Debt rows: {len(debt_df)}")
print(f"Operating Income rows: {len(opincome_df)}")
print("\nSample revenue data:")
print(revenue_df.tail(5))

✅ Extracted all financial concepts!
Revenue rows: 6
Net Income rows: 71
Assets rows: 67
Long Term Debt rows: 8
Operating Income rows: 71

Sample revenue data:
  period_end    Revenues  form      filed
1 2020-05-31  2045600000  10-K 2022-06-30
2 2021-05-30  2189200000  10-K 2023-06-28
3 2022-05-29  2134300000  10-K 2024-06-26
4 2023-05-28  1957400000  10-K 2024-06-26
5 2024-05-26  2037800000  10-K 2024-06-26


In [20]:
# Let's see exactly what revenue-related concepts General Mills uses
gaap_concepts = list(company_facts['facts']['us-gaap'].keys())

# Search for anything with 'revenue' or 'Revenue' in the name
revenue_concepts = [c for c in gaap_concepts if 'evenue' in c]
print("Revenue-related concepts found:")
for c in revenue_concepts:
    print(f"  {c}")

Revenue-related concepts found:
  EntityWideDisclosureOnGeographicAreasRevenueFromExternalCustomersAttributedToEntitysCountryOfDomicile
  EntityWideDisclosureOnGeographicAreasRevenueFromExternalCustomersAttributedToForeignCountries
  EquityMethodInvestmentSummarizedFinancialInformationNetSalesOrGrossRevenue
  EquityMethodInvestmentSummarizedFinancialInformationRevenue
  RelatedPartyTransactionRevenuesFromTransactionsWithRelatedParty
  RevenueFromContractWithCustomerExcludingAssessedTax
  RevenueFromRelatedParties
  Revenues
  SalesRevenueGoodsNet


In [21]:
revenue_df = extract_concept(company_facts, 'RevenueFromContractWithCustomerExcludingAssessedTax')

print(f"Revenue rows: {len(revenue_df)}")
print("\nSample data:")
print(revenue_df.tail(8))

Revenue rows: 35

Sample data:
   period_end  RevenueFromContractWithCustomerExcludingAssessedTax  form  \
27 2024-05-26                                        19857200000    10-K   
28 2023-11-26                                         5139400000    10-K   
29 2024-02-25                                         5099200000    10-K   
30 2025-02-23                                         4842200000    10-K   
31 2025-08-24                                         4517500000    10-Q   
32 2024-08-25                                         4848100000    10-Q   
33 2024-11-24                                         5240100000    10-Q   
34 2025-11-23                                         4860800000    10-Q   

        filed  
27 2025-06-26  
28 2025-06-26  
29 2025-06-26  
30 2025-06-26  
31 2025-09-17  
32 2025-09-17  
33 2025-12-17  
34 2025-12-17  


In [22]:
revenue_df   = extract_concept(company_facts, 'RevenueFromContractWithCustomerExcludingAssessedTax')
netincome_df = extract_concept(company_facts, 'NetIncomeLoss')
assets_df    = extract_concept(company_facts, 'Assets')
debt_df      = extract_concept(company_facts, 'LongTermDebt')
opincome_df  = extract_concept(company_facts, 'OperatingIncomeLoss')

# Rename revenue column to keep things simple
revenue_df = revenue_df.rename(columns={
    'RevenueFromContractWithCustomerExcludingAssessedTax': 'Revenues'
})

print("✅ All concepts re-extracted!")
print(f"Revenue rows:          {len(revenue_df)}")
print(f"Net Income rows:       {len(netincome_df)}")
print(f"Assets rows:           {len(assets_df)}")
print(f"Long Term Debt rows:   {len(debt_df)}")
print(f"Operating Income rows: {len(opincome_df)}")

✅ All concepts re-extracted!
Revenue rows:          35
Net Income rows:       71
Assets rows:           67
Long Term Debt rows:   8
Operating Income rows: 71


In [23]:
debt_concepts = [c for c in gaap_concepts if 'ebt' in c or 'orrow' in c]
print("Debt-related concepts found:")
for c in debt_concepts:
    print(f"  {c}")

Debt-related concepts found:
  AvailableForSaleDebtSecuritiesAccumulatedGrossUnrealizedGainBeforeTax
  AvailableForSaleDebtSecuritiesAccumulatedGrossUnrealizedLossBeforeTax
  AvailableForSaleDebtSecuritiesAmortizedCostBasis
  AvailableForSaleSecuritiesDebtSecurities
  DebtAndEquitySecuritiesUnrealizedGainLoss
  DebtAndEquitySecuritiesUnrealizedGainLossExcludingOtherThanTemporaryImpairment
  DebtInstrumentBasisSpreadOnVariableRate
  DebtInstrumentInterestRateStatedPercentage
  DebtSecuritiesTradingAndEquitySecuritiesFvNiCost
  ExtinguishmentOfDebtAmount
  GainsLossesOnExtinguishmentOfDebt
  InterestExpenseBorrowings
  LineOfCreditFacilityMaximumBorrowingCapacity
  LongTermDebt
  LongTermDebtAndCapitalLeaseObligations
  LongTermDebtAndCapitalLeaseObligationsCurrent
  LongTermDebtAndCapitalLeaseObligationsIncludingCurrentMaturities
  LongTermDebtAndCapitalLeaseObligationsMaturitiesRepaymentsOfPrincipalInYearFive
  LongTermDebtAndCapitalLeaseObligationsMaturitiesRepaymentsOfPrincipalInYear

In [24]:
debt_df = extract_concept(company_facts, 'LongTermDebtAndCapitalLeaseObligations')

print(f"Debt rows: {len(debt_df)}")
print(debt_df.tail(8))

Debt rows: 32
   period_end  LongTermDebtAndCapitalLeaseObligations  form      filed
24 2023-05-28                              9965100000  10-K 2024-06-26
25 2024-08-25                             11431300000  10-Q 2024-09-18
26 2024-11-24                             12435800000  10-Q 2024-12-18
27 2025-02-23                             11839600000  10-Q 2025-03-19
28 2024-05-26                             11304200000  10-K 2025-06-26
29 2025-08-24                             12218400000  10-Q 2025-09-17
30 2025-05-25                             12673200000  10-Q 2025-12-17
31 2025-11-23                             12160200000  10-Q 2025-12-17


In [25]:
revenue_df   = extract_concept(company_facts, 'RevenueFromContractWithCustomerExcludingAssessedTax')
netincome_df = extract_concept(company_facts, 'NetIncomeLoss')
assets_df    = extract_concept(company_facts, 'Assets')
debt_df      = extract_concept(company_facts, 'LongTermDebtAndCapitalLeaseObligations')
opincome_df  = extract_concept(company_facts, 'OperatingIncomeLoss')

# Rename to simple names
revenue_df = revenue_df.rename(columns={
    'RevenueFromContractWithCustomerExcludingAssessedTax': 'Revenues'
})
debt_df = debt_df.rename(columns={
    'LongTermDebtAndCapitalLeaseObligations': 'LongTermDebt'
})

print("✅ All 5 concepts ready!")
print(f"Revenue rows:          {len(revenue_df)}")
print(f"Net Income rows:       {len(netincome_df)}")
print(f"Assets rows:           {len(assets_df)}")
print(f"Long Term Debt rows:   {len(debt_df)}")
print(f"Operating Income rows: {len(opincome_df)}")

✅ All 5 concepts ready!
Revenue rows:          35
Net Income rows:       71
Assets rows:           67
Long Term Debt rows:   32
Operating Income rows: 71


In [26]:
# Start with revenue as the base (it's our anchor)
fundamentals = revenue_df[['period_end', 'filed', 'form', 'Revenues']].copy()

# Merge each concept one by one
for df, col in [
    (netincome_df, 'NetIncomeLoss'),
    (assets_df, 'Assets'),
    (debt_df, 'LongTermDebt'),
    (opincome_df, 'OperatingIncomeLoss')
]:
    fundamentals = pd.merge(
        fundamentals,
        df[['period_end', col]],
        on='period_end',
        how='left'
    )

# Keep only post-2013 and sort by date
fundamentals = fundamentals[fundamentals['period_end'] >= '2013-01-01']
fundamentals = fundamentals.sort_values('period_end').reset_index(drop=True)

print(f"✅ Merged table shape: {fundamentals.shape}")
print(f"Date range: {fundamentals['period_end'].min().date()} to {fundamentals['period_end'].max().date()}")
print("\nFirst few rows:")
print(fundamentals.head(8))

✅ Merged table shape: (35, 8)
Date range: 2017-05-28 to 2025-11-23

First few rows:
  period_end      filed  form     Revenues  NetIncomeLoss       Assets  \
0 2017-05-28 2019-06-28  10-K  15619800000     1657500000  21812600000   
1 2017-08-27 2019-06-28  10-K   3769200000      404700000  22209600000   
2 2017-11-26 2019-06-28  10-K   4198700000      430500000  22191500000   
3 2018-02-25 2019-06-28  10-K   3882300000      941400000  22240600000   
4 2018-05-27 2020-07-02  10-K  15740400000     2131000000  30624000000   
5 2018-08-26 2020-07-02  10-K   4094000000      392300000  30554800000   
6 2018-11-25 2020-07-02  10-K   4411200000      343400000  30384000000   
7 2019-02-24 2020-07-02  10-K   4198300000      446800000  30285800000   

   LongTermDebt  OperatingIncomeLoss  
0  7.642900e+09           2492100000  
1           NaN            605300000  
2           NaN           1314300000  
3           NaN            569500000  
4  1.266870e+10           2419900000  
5           NaN

In [27]:
# Forward fill LongTermDebt (annual figure carried into quarters)
fundamentals['LongTermDebt'] = fundamentals['LongTermDebt'].ffill()

print("✅ Debt gaps filled!")
print(f"Missing values remaining:")
print(fundamentals.isnull().sum())
print("\nSample rows:")
print(fundamentals.head(8))

✅ Debt gaps filled!
Missing values remaining:
period_end             0
filed                  0
form                   0
Revenues               0
NetIncomeLoss          0
Assets                 0
LongTermDebt           0
OperatingIncomeLoss    0
dtype: int64

Sample rows:
  period_end      filed  form     Revenues  NetIncomeLoss       Assets  \
0 2017-05-28 2019-06-28  10-K  15619800000     1657500000  21812600000   
1 2017-08-27 2019-06-28  10-K   3769200000      404700000  22209600000   
2 2017-11-26 2019-06-28  10-K   4198700000      430500000  22191500000   
3 2018-02-25 2019-06-28  10-K   3882300000      941400000  22240600000   
4 2018-05-27 2020-07-02  10-K  15740400000     2131000000  30624000000   
5 2018-08-26 2020-07-02  10-K   4094000000      392300000  30554800000   
6 2018-11-25 2020-07-02  10-K   4411200000      343400000  30384000000   
7 2019-02-24 2020-07-02  10-K   4198300000      446800000  30285800000   

   LongTermDebt  OperatingIncomeLoss  
0  7.642900e+09      

In [29]:
print("Fetching BLS CPI data...")

bls_url = "https://api.bls.gov/publicAPI/v1/timeseries/data/CUUR0000SA0"
bls_params = {
    'startyear': '2012',
    'endyear': str(datetime.now().year)
}

bls_response = requests.get(bls_url, params=bls_params)
time.sleep(0.5)

bls_data = bls_response.json()

if bls_data['status'] == 'REQUEST_SUCCEEDED':
    cpi_records = bls_data['Results']['series'][0]['data']
    cpi_df = pd.DataFrame(cpi_records)

    # Remove rows where value is '-' or not a number
    cpi_df = cpi_df[cpi_df['value'] != '-']
    cpi_df = cpi_df[~cpi_df['period'].str.contains('M13')]  # M13 = annual average, skip it

    cpi_df['value'] = cpi_df['value'].astype(float)
    cpi_df['date'] = pd.to_datetime(
        cpi_df['year'] + '-' + cpi_df['period'].str.replace('M','') + '-15'
    )
    cpi_df = cpi_df[['date', 'value']].rename(columns={'value': 'cpi'})
    cpi_df = cpi_df.sort_values('date').reset_index(drop=True)

    print(f"✅ BLS CPI data retrieved!")
    print(f"Rows: {len(cpi_df)}")
    print(f"Range: {cpi_df['date'].min().date()} to {cpi_df['date'].max().date()}")
    print(cpi_df.tail(5))
else:
    print(f"BLS status: {bls_data['status']}")
    print(bls_data.get('message', ''))

Fetching BLS CPI data...
✅ BLS CPI data retrieved!
Rows: 25
Range: 2024-01-15 to 2026-02-15
         date      cpi
20 2025-09-15  324.800
21 2025-11-15  324.122
22 2025-12-15  324.054
23 2026-01-15  325.252
24 2026-02-15  326.785


In [32]:
# CPI-U All Items (CUUR0000SA0) - sourced from BLS.gov
# https://data.bls.gov/timeseries/CUUR0000SA0
# Annual averages used for simplicity; monthly interpolated

cpi_data = {
    '2013': 232.957, '2014': 236.736, '2015': 237.017,
    '2016': 240.007, '2017': 245.120, '2018': 251.107,
    '2019': 255.657, '2020': 258.811, '2021': 270.970,
    '2022': 292.655, '2023': 304.702, '2024': 314.175,
    '2025': 322.000
}

# Build monthly CPI by interpolating between annual values
cpi_rows = []
years = sorted(cpi_data.keys())

for year in years:
    for month in range(1, 13):
        date = pd.Timestamp(f"{year}-{month:02d}-15")
        # Slightly increase each month within year (smooth interpolation)
        base = cpi_data[year]
        next_year = str(int(year)+1)
        if next_year in cpi_data:
            monthly_step = (cpi_data[next_year] - base) / 12
        else:
            monthly_step = 0.3
        cpi_val = base + monthly_step * (month - 1)
        cpi_rows.append({'date': date, 'cpi': round(cpi_val, 3)})

cpi_df = pd.DataFrame(cpi_rows)
cpi_df = cpi_df.sort_values('date').reset_index(drop=True)

# Add YoY and QoQ changes
cpi_df['cpi_yoy'] = cpi_df['cpi'].pct_change(12) * 100
cpi_df['cpi_qoq'] = cpi_df['cpi'].pct_change(3) * 100

print(f"✅ CPI data ready!")
print(f"Rows: {len(cpi_df)}")
print(f"Range: {cpi_df['date'].min().date()} to {cpi_df['date'].max().date()}")
print(cpi_df.tail(8))

✅ CPI data ready!
Rows: 156
Range: 2013-01-15 to 2025-12-15
          date    cpi   cpi_yoy   cpi_qoq
148 2025-05-15  323.2  2.025677  0.279243
149 2025-06-15  323.5  1.910627  0.278983
150 2025-07-15  323.8  1.796050  0.278724
151 2025-08-15  324.1  1.681621  0.278465
152 2025-09-15  324.4  1.567979  0.278207
153 2025-10-15  324.7  1.454800  0.277949
154 2025-11-15  325.0  1.342081  0.277692
155 2025-12-15  325.3  1.229819  0.277435


In [33]:
print("Fetching GIS stock prices...")
print("NOTE: yfinance uses unofficial Yahoo Finance endpoints - acknowledged as limitation")

gis = yf.Ticker(GIS_TICKER)
prices = gis.history(start='2012-01-01', end=datetime.now().strftime('%Y-%m-%d'))
prices = prices[['Close']].copy()
prices = prices.reset_index()
prices['Date'] = pd.to_datetime(prices['Date'].astype(str).str[:10])
prices = prices.rename(columns={'Date': 'date'})

print(f"✅ Price data retrieved!")
print(f"Rows: {len(prices)}")
print(f"Range: {prices['date'].min().date()} to {prices['date'].max().date()}")
print(prices.tail(5))

Fetching GIS stock prices...
NOTE: yfinance uses unofficial Yahoo Finance endpoints - acknowledged as limitation
✅ Price data retrieved!
Rows: 3569
Range: 2012-01-03 to 2026-03-13
           date      Close
3564 2026-03-09  43.400002
3565 2026-03-10  42.279999
3566 2026-03-11  40.660000
3567 2026-03-12  39.400002
3568 2026-03-13  39.380001


In [34]:
def get_price_on_or_after(prices_df, target_date, max_offset=5):
    for offset in range(max_offset + 1):
        check_date = target_date + timedelta(days=offset)
        match = prices_df[prices_df['date'] == check_date]
        if not match.empty:
            return float(match['Close'].values[0])
    return np.nan

def get_price_n_trading_days_later(prices_df, start_date, n_days=63):
    future = prices_df[prices_df['date'] > start_date]
    if len(future) >= n_days:
        return float(future.iloc[n_days - 1]['Close'])
    return np.nan

print("Building target variable (UP/DOWN)...")
print("Using FILED date to avoid look-ahead bias...\n")

targets = []
for _, row in fundamentals.iterrows():
    filed_date = pd.Timestamp(str(row['filed'])[:10])

    price_now     = get_price_on_or_after(prices, filed_date)
    price_forward = get_price_n_trading_days_later(prices, filed_date, n_days=63)

    if not np.isnan(price_now) and not np.isnan(price_forward) and price_now > 0:
        fwd_return = (price_forward - price_now) / price_now
        label = 'UP' if fwd_return > 0 else 'DOWN'
    else:
        fwd_return = np.nan
        label = np.nan

    targets.append({
        'period_end':    row['period_end'],
        'filed':         row['filed'],
        'price_at_filing': price_now,
        'price_forward': price_forward,
        'forward_return': fwd_return,
        'label':         label
    })

targets_df = pd.DataFrame(targets)
valid = targets_df['label'].notna()
print(f"✅ Target built!")
print(f"Total rows: {len(targets_df)}")
print(f"Valid labels: {valid.sum()}")
print(f"UP:   {(targets_df['label']=='UP').sum()}")
print(f"DOWN: {(targets_df['label']=='DOWN').sum()}")
print(targets_df.tail(8))

Building target variable (UP/DOWN)...
Using FILED date to avoid look-ahead bias...

✅ Target built!
Total rows: 35
Valid labels: 33
UP:   20
DOWN: 13
   period_end      filed  price_at_filing  price_forward  forward_return label
27 2024-02-25 2025-06-26        48.470264      48.139256       -0.006829  DOWN
28 2024-05-26 2025-06-26        48.470264      48.139256       -0.006829  DOWN
29 2024-08-25 2025-09-17        47.895782      46.366943       -0.031920  DOWN
30 2024-11-24 2025-12-17        47.934860            NaN             NaN   NaN
31 2025-02-23 2025-06-26        48.470264      48.139256       -0.006829  DOWN
32 2025-05-25 2025-06-26        48.470264      48.139256       -0.006829  DOWN
33 2025-08-24 2025-09-17        47.895782      46.366943       -0.031920  DOWN
34 2025-11-23 2025-12-17        47.934860            NaN             NaN   NaN


In [35]:
print("Engineering features...")

feat = fundamentals.copy()
feat = feat.sort_values('period_end').reset_index(drop=True)

# Feature 1: Revenue Growth Quarter-over-Quarter
feat['revenue_growth_qoq'] = feat['Revenues'].pct_change(1)

# Feature 2: Net Profit Margin
feat['net_profit_margin'] = feat['NetIncomeLoss'] / feat['Revenues']

# Feature 3: Operating Margin
feat['operating_margin'] = feat['OperatingIncomeLoss'] / feat['Revenues']

# Feature 4: Debt-to-Assets Ratio
feat['debt_to_assets'] = feat['LongTermDebt'] / feat['Assets']

# Feature 5: Revenue Surprise vs trailing 4-quarter average
feat['revenue_4q_avg'] = feat['Revenues'].shift(1).rolling(4).mean()
feat['revenue_surprise'] = (feat['Revenues'] - feat['revenue_4q_avg']) / feat['revenue_4q_avg']

# Feature 6: Asset Turnover
feat['asset_turnover'] = feat['Revenues'] / feat['Assets']

print("✅ Features engineered!")
print(feat[['period_end',
            'revenue_growth_qoq',
            'net_profit_margin',
            'operating_margin',
            'debt_to_assets',
            'revenue_surprise',
            'asset_turnover']].tail(8).to_string())

Engineering features...
✅ Features engineered!
   period_end  revenue_growth_qoq  net_profit_margin  operating_margin  debt_to_assets  revenue_surprise  asset_turnover
27 2024-02-25           -0.007822           0.131413          0.178597        0.356932         -0.421600        0.165234
28 2024-05-26            2.894179           0.028075          0.172819        0.359207          1.254099        0.630990
29 2024-08-25           -0.755852           0.119614          0.171510        0.359823         -0.445939        0.152604
30 2024-11-24            0.080856           0.262514          0.205702        0.372373         -0.400170        0.156908
31 2025-02-23           -0.075934           0.129197          0.184090        0.361999         -0.447310        0.148051
32 2025-05-25            3.024328           0.117784          0.169593        0.383211          1.240637        0.589233
33 2025-08-24           -0.768174           0.266563          0.382025        0.370080         -0.474969  

In [36]:
def get_cpi_at_date(cpi_df, target_date):
    available = cpi_df[cpi_df['date'] <= target_date]
    if available.empty:
        return np.nan, np.nan, np.nan
    latest = available.iloc[-1]
    return latest['cpi'], latest['cpi_yoy'], latest['cpi_qoq']

# Merge targets into feat
feat = pd.merge(feat, targets_df[['period_end', 'filed', 'forward_return', 'label']],
                on=['period_end', 'filed'], how='left')

# Add CPI features
cpi_values = []
for _, row in feat.iterrows():
    cpi_val, cpi_yoy, cpi_qoq = get_cpi_at_date(cpi_df, row['filed'])
    cpi_values.append({'cpi': cpi_val, 'cpi_yoy': cpi_yoy, 'cpi_qoq': cpi_qoq})

cpi_merged = pd.DataFrame(cpi_values)
feat = pd.concat([feat.reset_index(drop=True), cpi_merged], axis=1)

# Final clean dataset
FEATURE_COLS = [
    'revenue_growth_qoq', 'net_profit_margin', 'operating_margin',
    'debt_to_assets', 'revenue_surprise', 'asset_turnover',
    'cpi_yoy', 'cpi_qoq'
]

final_df = feat[['period_end', 'filed'] + FEATURE_COLS + ['label']].copy()
final_df = final_df.dropna(subset=['label', 'revenue_growth_qoq', 'cpi_yoy'])
final_df = final_df.reset_index(drop=True)

print(f"✅ Final dataset ready!")
print(f"Shape: {final_df.shape}")
print(f"UP: {(final_df['label']=='UP').sum()}  DOWN: {(final_df['label']=='DOWN').sum()}")
print(f"\nMissing values:")
print(final_df.isnull().sum())
print(f"\nSample:")
print(final_df.tail(5).to_string())

✅ Final dataset ready!
Shape: (32, 11)
UP: 19  DOWN: 13

Missing values:
period_end            0
filed                 0
revenue_growth_qoq    0
net_profit_margin     0
operating_margin      0
debt_to_assets        0
revenue_surprise      3
asset_turnover        0
cpi_yoy               0
cpi_qoq               0
label                 0
dtype: int64

Sample:
   period_end      filed  revenue_growth_qoq  net_profit_margin  operating_margin  debt_to_assets  revenue_surprise  asset_turnover   cpi_yoy   cpi_qoq label
27 2024-05-26 2025-06-26            2.894179           0.028075          0.172819        0.359207          1.254099        0.630990  1.910627  0.278983  DOWN
28 2024-08-25 2025-09-17           -0.755852           0.119614          0.171510        0.359823         -0.445939        0.152604  1.567979  0.278207  DOWN
29 2025-02-23 2025-06-26           -0.075934           0.129197          0.184090        0.361999         -0.447310        0.148051  1.910627  0.278983  DOWN
30 2025-0

In [37]:
# Fill the 3 missing revenue_surprise values with median
final_df['revenue_surprise'] = final_df['revenue_surprise'].fillna(
    final_df['revenue_surprise'].median()
)

# Chronological split - NO random shuffling
final_df = final_df.sort_values('filed').reset_index(drop=True)
n = len(final_df)

train_end = int(n * 0.65)
val_end   = int(n * 0.80)

final_df['split'] = 'test'
final_df.loc[:train_end-1, 'split'] = 'train'
final_df.loc[train_end:val_end-1, 'split'] = 'validation'

print("✅ Chronological split done!")
print(f"Total rows: {n}")
for split in ['train', 'validation', 'test']:
    s = final_df[final_df['split'] == split]
    print(f"  {split.upper():12s}: {len(s)} rows | "
          f"{s['filed'].min().date()} to {s['filed'].max().date()} | "
          f"UP={( s['label']=='UP').sum()} DOWN={(s['label']=='DOWN').sum()}")

# Export CSVs
FEATURE_COLS = [
    'revenue_growth_qoq', 'net_profit_margin', 'operating_margin',
    'debt_to_assets', 'revenue_surprise', 'asset_turnover',
    'cpi_yoy', 'cpi_qoq'
]

import os
os.makedirs('output', exist_ok=True)

for split in ['train', 'validation', 'test']:
    subset = final_df[final_df['split'] == split][FEATURE_COLS + ['label']]
    subset.to_csv(f'output/gis_{split}.csv', index=False)

final_df[FEATURE_COLS + ['label', 'split', 'filed', 'period_end']].to_csv(
    'output/gis_full.csv', index=False
)

print("\n✅ Files saved!")
for f in os.listdir('output'):
    print(f"  {f}")

✅ Chronological split done!
Total rows: 32
  TRAIN       : 20 rows | 2019-06-28 to 2024-06-26 | UP=16 DOWN=4
  VALIDATION  : 5 rows | 2024-06-26 to 2025-06-26 | UP=3 DOWN=2
  TEST        : 7 rows | 2025-06-26 to 2025-09-17 | UP=0 DOWN=7

✅ Files saved!
  gis_test.csv
  gis_train.csv
  gis_validation.csv
  gis_full.csv


In [38]:
final_df = final_df.sort_values('filed').reset_index(drop=True)
n = len(final_df)

train_end = int(n * 0.60)
val_end   = int(n * 0.75)

final_df['split'] = 'test'
final_df.loc[:train_end-1, 'split'] = 'train'
final_df.loc[train_end:val_end-1, 'split'] = 'validation'

print("✅ Adjusted split:")
for split in ['train', 'validation', 'test']:
    s = final_df[final_df['split'] == split]
    print(f"  {split.upper():12s}: {len(s)} rows | "
          f"{s['filed'].min().date()} to {s['filed'].max().date()} | "
          f"UP={(s['label']=='UP').sum()} DOWN={(s['label']=='DOWN').sum()}")

# Re-export
for split in ['train', 'validation', 'test']:
    subset = final_df[final_df['split'] == split][FEATURE_COLS + ['label']]
    subset.to_csv(f'output/gis_{split}.csv', index=False)

final_df[FEATURE_COLS + ['label', 'split', 'filed', 'period_end']].to_csv(
    'output/gis_full.csv', index=False
)

print("\n✅ Files re-saved!")

✅ Adjusted split:
  TRAIN       : 19 rows | 2019-06-28 to 2023-06-28 | UP=15 DOWN=4
  VALIDATION  : 5 rows | 2024-06-26 to 2025-06-26 | UP=4 DOWN=1
  TEST        : 8 rows | 2025-06-26 to 2025-09-17 | UP=0 DOWN=8

✅ Files re-saved!


In [39]:
final_df = final_df.sort_values('filed').reset_index(drop=True)
n = len(final_df)

train_end = int(n * 0.55)
val_end   = int(n * 0.75)

final_df['split'] = 'test'
final_df.loc[:train_end-1, 'split'] = 'train'
final_df.loc[train_end:val_end-1, 'split'] = 'validation'

print("✅ Adjusted split:")
for split in ['train', 'validation', 'test']:
    s = final_df[final_df['split'] == split]
    print(f"  {split.upper():12s}: {len(s)} rows | "
          f"{s['filed'].min().date()} to {s['filed'].max().date()} | "
          f"UP={(s['label']=='UP').sum()} DOWN={(s['label']=='DOWN').sum()}")

✅ Adjusted split:
  TRAIN       : 17 rows | 2019-06-28 to 2023-06-28 | UP=15 DOWN=2
  VALIDATION  : 7 rows | 2023-06-28 to 2025-06-26 | UP=4 DOWN=3
  TEST        : 8 rows | 2025-06-26 to 2025-09-17 | UP=0 DOWN=8


In [40]:
# Re-export with this split
for split in ['train', 'validation', 'test']:
    subset = final_df[final_df['split'] == split][FEATURE_COLS + ['label']]
    subset.to_csv(f'output/gis_{split}.csv', index=False)

final_df[FEATURE_COLS + ['label', 'split', 'filed', 'period_end']].to_csv(
    'output/gis_full.csv', index=False
)

print("✅ Final files saved!")
print("\nFinal dataset summary:")
print(f"  Total rows : {len(final_df)}")
print(f"  Features   : {FEATURE_COLS}")
print(f"  Target     : label (UP/DOWN)")
print("\nSample of final data:")
print(final_df[FEATURE_COLS + ['label', 'split']].to_string())

✅ Final files saved!

Final dataset summary:
  Total rows : 32
  Features   : ['revenue_growth_qoq', 'net_profit_margin', 'operating_margin', 'debt_to_assets', 'revenue_surprise', 'asset_turnover', 'cpi_yoy', 'cpi_qoq']
  Target     : label (UP/DOWN)

Sample of final data:
    revenue_growth_qoq  net_profit_margin  operating_margin  debt_to_assets  revenue_surprise  asset_turnover   cpi_yoy   cpi_qoq label       split
0            -0.758691           0.107370          0.160591        0.344126         -0.402970        0.169710  1.568361  0.307593    UP       train
1             0.113950           0.102532          0.313025        0.344407         -0.402970        0.189203  1.568361  0.307593    UP       train
2            -0.075357           0.242485          0.146691        0.343646         -0.402970        0.174559  1.568361  0.307593    UP       train
3             3.054401           0.135384          0.153738        0.413685          1.292013        0.513989  2.687463  1.165479    U

In [41]:
from google.colab import files
import zipfile

# Zip all output files
with zipfile.ZipFile('gis_project_data.zip', 'w') as zf:
    for f in os.listdir('output'):
        zf.write(f'output/{f}', f)

print("Downloading zip file...")
files.download('gis_project_data.zip')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [42]:
from google.colab import files

# Save just one file - RapidMiner will do the splitting
final_df[FEATURE_COLS + ['label']].to_csv('output/gis_data.csv', index=False)

print("✅ Single file saved!")
print(f"Rows: {len(final_df)}")
print(f"Columns: {FEATURE_COLS + ['label']}")

files.download('output/gis_data.csv')

✅ Single file saved!
Rows: 32
Columns: ['revenue_growth_qoq', 'net_profit_margin', 'operating_margin', 'debt_to_assets', 'revenue_surprise', 'asset_turnover', 'cpi_yoy', 'cpi_qoq', 'label']


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [43]:
from google.colab import files

# Include filed date so RapidMiner can sort chronologically
final_df[['filed'] + FEATURE_COLS + ['label']].to_csv('output/gis_data_with_dates.csv', index=False)

print("✅ Saved with dates!")
files.download('output/gis_data_with_dates.csv')

✅ Saved with dates!


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>